# 06 · Community Behavior

HN has a power-law karma distribution, dead hours, and a specific taxonomy of post types. Here we quantify the inequality, identify the structural patterns, and find the posts that generate debate without generating upvotes.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.loader import db
from src.viz import set_style, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        score,
        comment_count,
        author,
        HOUR(posted_at) AS hour_utc,
        DAYOFWEEK(posted_at) AS dow,
        YEAR(posted_at) AS year
    FROM stories
    WHERE score IS NOT NULL AND year BETWEEN 2008 AND 2024
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Score inequality — Gini coefficient

In [3]:
def gini(arr: np.ndarray) -> float:
    arr = np.sort(arr)
    n = len(arr)
    return (2 * np.sum(np.arange(1, n + 1) * arr) / (n * arr.sum())) - (n + 1) / n

scores = stories['score'].dropna().values
g = gini(scores)
print(f'Gini coefficient of HN story scores: {g:.3f}')
print('(0 = perfect equality, 1 = one story gets all points)')

# Lorenz curve
sorted_scores = np.sort(scores)
cumulative_scores = np.cumsum(sorted_scores) / sorted_scores.sum()
n = len(sorted_scores)
cumulative_pop = np.arange(1, n + 1) / n

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(cumulative_pop * 100, cumulative_scores * 100, color='#e8604c', linewidth=2, label=f'HN stories (Gini={g:.2f})')
ax.plot([0, 100], [0, 100], 'k--', linewidth=1, alpha=0.5, label='Perfect equality')
ax.fill_between(cumulative_pop * 100, cumulative_scores * 100, cumulative_pop * 100, alpha=0.15, color='#e8604c')
ax.set_title('Lorenz curve of HN story scores', fontsize=13, fontweight='bold')
ax.set_xlabel('Cumulative % of stories (sorted by score)')
ax.set_ylabel('Cumulative % of total points')
ax.legend()
plt.tight_layout()
save(fig, '../data/fig_lorenz.png')
plt.show()

top1_pct = np.percentile(scores, 99)
top1_share = scores[scores >= top1_pct].sum() / scores.sum() * 100
print(f'Top 1% of stories (score ≥ {top1_pct:.0f}) capture {top1_share:.1f}% of all points')

Gini coefficient of HN story scores: 0.845
(0 = perfect equality, 1 = one story gets all points)


/Users/zascosium/Documents/github_proj_cv/hackermining/notebooks/../src/viz.py:125: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=BG)


Top 1% of stories (score ≥ 259) capture 32.6% of all points


## Ask HN vs Show HN vs plain links

In [4]:
stories['post_type'] = 'Link'
stories.loc[stories['title'].str.startswith('Ask HN:'), 'post_type'] = 'Ask HN'
stories.loc[stories['title'].str.startswith('Show HN:'), 'post_type'] = 'Show HN'
stories.loc[stories['title'].str.startswith('Tell HN:'), 'post_type'] = 'Tell HN'

type_stats = (
    stories.groupby('post_type')
    .agg(
        count=('score', 'count'),
        median_score=('score', 'median'),
        median_comments=('comment_count', 'median'),
    )
    .sort_values('count', ascending=False)
)
display(type_stats)

,count,median_score,median_comments
post_type,,,
Link,4690333,2.0,0.0
Ask HN,182897,3.0,2.0
Show HN,153527,3.0,0.0
Tell HN,4892,5.0,3.0


## Power authors — top submitters of front-page stories

In [5]:
TOP_SCORE = 200
power = (
    stories[stories['score'] >= TOP_SCORE]
    .groupby('author')
    .agg(viral_posts=('score', 'count'), total_points=('score', 'sum'))
    .sort_values('viral_posts', ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(power.index[::-1], power['viral_posts'][::-1], color='#e8604c', edgecolor='white')
ax.set_title(f'Top submitters of stories scoring ≥ {TOP_SCORE}', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of high-score posts')
plt.tight_layout()
save(fig, '../data/fig_power_authors.png')
plt.show()